Import all necessary libraries.

In [1]:
import pandas as pd
import numpy as np
import os
import wandb
import random
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader
from imblearn.metrics import geometric_mean_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

import sys
sys.path.append(os.path.join(os.getcwd(), '../src'))

from transforms.feature_engineering_classification import add_all_features, filter_business_hours, entries_per_day_per_site
from transforms.feature_engineering_classification import (
    CONTINUOUS_FEATURE_COLUMNS,
    CATEGORICAL_FEATURE_COLUMNS,
    CYCLIC_FEATURE_COLUMNS,
    TARGET_COLUMN
)
from evaluation.comp_metrics import evaluate_all_metrics
from datasets.flextrack_dataset import FlextrackClassificationDataset

c:\Users\timon\.pyenv-win-venv\envs\aicomp\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\timon\.pyenv-win-venv\envs\aicomp\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because a

Set seed for reproducibility.

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
os.environ["WANDB_API_KEY"] = "3aaf9f796df65417b3f5f8560b43875171b55805"

wandb.login()

wandb: Currently logged in as: fabian-dubach (fabian-dubach-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
# just use regression data and remove DR-Flags and DR-Capacity
df_train = pd.read_csv(os.path.normpath(os.path.join(os.getcwd(), '../data/regression/regression-train.csv')))
df_test = pd.read_csv(os.path.normpath(os.path.join(os.getcwd(), '../data/regression/regression-test.csv')))

In [6]:
print(f"Dataset shape: {df_train.shape}")
print(f"\nColumns: {df_train.columns.tolist()}")

Dataset shape: (105120, 7)

Columns: ['Site', 'Timestamp_Local', 'Dry_Bulb_Temperature_C', 'Global_Horizontal_Radiation_W/m2', 'Building_Power_kW', 'Demand_Response_Flag', 'Demand_Response_Capacity_kW']


# Feature Engineering

We use same feature engineering as in regression task but remove the irrelevant features.

In [7]:
df_train = add_all_features(df_train)

df_train = filter_business_hours(df_train)

ENTRIES_PER_DAY = entries_per_day_per_site(df_train)

In [8]:
print(f"Dataset shape: {df_train.shape}")
print(f"\nColumns: {df_train.columns.tolist()}")

Dataset shape: (58035, 40)

Columns: ['Site', 'Timestamp_Local', 'Dry_Bulb_Temperature_C', 'Global_Horizontal_Radiation_W/m2', 'Building_Power_kW', 'Demand_Response_Flag', 'Demand_Response_Capacity_kW', 'hour', 'minute', 'day_of_week', 'is_weekend', 'is_holiday', 'month_sin', 'month_cos', 'Building_Power_kW_diff_15min', 'Building_Power_kW_diff_1h', 'Building_Power_kW_diff_1d', 'Dry_Bulb_Temperature_C_diff_15min', 'Global_Horizontal_Radiation_W/m2_diff_15min', 'Building_Power_kW_rolling_mean_1h', 'Building_Power_kW_rolling_mean_2h', 'Building_Power_kW_rolling_mean_1d', 'Building_Power_kW_rolling_std_1h', 'Building_Power_kW_rolling_std_2h', 'Building_Power_kW_rolling_std_1d', 'Building_Power_kW_rolling_min_1h', 'Building_Power_kW_rolling_min_2h', 'Building_Power_kW_rolling_max_1h', 'Building_Power_kW_rolling_max_2h', 'minute_0', 'minute_15', 'minute_30', 'minute_45', 'day_of_week_0', 'day_of_week_1', 'day_of_week_2', 'day_of_week_3', 'day_of_week_4', 'day_of_week_5', 'day_of_week_6']


# Split sites

In [9]:
def count_sites(df):

    counter_site_a = 0
    counter_site_b = 0
    counter_site_c = 0
    counter_site_d = 0
    counter_site_e = 0

    for i in df['Site']:
        if i == 'siteA':
            counter_site_a += 1
        elif i == 'siteB':
            counter_site_b += 1
        elif i == 'siteC':
            counter_site_c += 1
        elif i == 'siteD':
            counter_site_d += 1
        elif i == 'siteE':
            counter_site_e += 1
    
    return counter_site_a, counter_site_b, counter_site_c, counter_site_d, counter_site_e

In [10]:
df_train_site_a = df_train[0:19345]
count_sites(df_train_site_a)

df_train_site_b = df_train[19345:38690]
count_sites(df_train_site_b)

df_train_site_c = df_train[38690:58035]
count_sites(df_train_site_c)

(0, 0, 19345, 0, 0)

In [11]:
X_continuous_site_a = df_train_site_a[CONTINUOUS_FEATURE_COLUMNS].values # Convert to numpy array
X_continuous_site_b = df_train_site_b[CONTINUOUS_FEATURE_COLUMNS].values # Convert to numpy array
X_continuous_site_c = df_train_site_c[CONTINUOUS_FEATURE_COLUMNS].values # Convert to numpy array

X_categorical_site_a = df_train_site_a[CATEGORICAL_FEATURE_COLUMNS].values # Convert to numpy array
X_categorical_site_b = df_train_site_b[CATEGORICAL_FEATURE_COLUMNS].values # Convert to numpy array
X_categorical_site_c = df_train_site_c[CATEGORICAL_FEATURE_COLUMNS].values # Convert to numpy array

X_cyclic_site_a = df_train_site_a[CYCLIC_FEATURE_COLUMNS].values # Convert to numpy array
X_cyclic_site_b = df_train_site_b[CYCLIC_FEATURE_COLUMNS].values # Convert to numpy array
X_cyclic_site_c = df_train_site_c[CYCLIC_FEATURE_COLUMNS].values # Convert to numpy array

y_site_a = df_train_site_a[TARGET_COLUMN].values.reshape(-1, 1) # Convert to numpy array and reshape
y_site_b = df_train_site_b[TARGET_COLUMN].values.reshape(-1, 1) # Convert to numpy array and reshape
y_site_c = df_train_site_c[TARGET_COLUMN].values.reshape(-1, 1) # Convert to numpy array and reshape

In [12]:
print(f"Continuous feature shape: {X_continuous_site_a.shape}")
print(f"Continuous feature shape: {X_continuous_site_b.shape}")
print(f"Continuous feature shape: {X_continuous_site_c.shape}")

print(f"Categorical feature shape: {X_categorical_site_a.shape}")
print(f"Categorical feature shape: {X_categorical_site_b.shape}")
print(f"Categorical feature shape: {X_categorical_site_c.shape}")

print(f"Cyclic feature shape: {X_cyclic_site_a.shape}")
print(f"Cyclic feature shape: {X_cyclic_site_b.shape}")
print(f"Cyclic feature shape: {X_cyclic_site_c.shape}")

print(f"Target shape: {y_site_a.shape}")
print(f"Target shape: {y_site_b.shape}")
print(f"Target shape: {y_site_c.shape}")

Continuous feature shape: (19345, 19)
Continuous feature shape: (19345, 19)
Continuous feature shape: (19345, 19)
Categorical feature shape: (19345, 13)
Categorical feature shape: (19345, 13)
Categorical feature shape: (19345, 13)
Cyclic feature shape: (19345, 2)
Cyclic feature shape: (19345, 2)
Cyclic feature shape: (19345, 2)
Target shape: (19345, 1)
Target shape: (19345, 1)
Target shape: (19345, 1)


In [13]:
y_site_a_unscaled = y_site_a.copy()

# Normalization

In [14]:
scaler_X_site_a = StandardScaler()
scaler_X_site_b = StandardScaler()
scaler_X_site_c = StandardScaler()

In [15]:
X_scaled_site_a = scaler_X_site_a.fit_transform(X_continuous_site_a)
X_scaled_site_b = scaler_X_site_b.fit_transform(X_continuous_site_b)
X_scaled_site_c = scaler_X_site_c.fit_transform(X_continuous_site_c)

Concatenate the unscaled and the scaled features together.

In [16]:
X_site_a = np.concatenate([X_scaled_site_a, X_categorical_site_a, X_cyclic_site_a], axis=1)
X_site_b = np.concatenate([X_scaled_site_b, X_categorical_site_b, X_cyclic_site_b], axis=1)
X_site_c = np.concatenate([X_scaled_site_c, X_categorical_site_c, X_cyclic_site_c], axis=1)

In [17]:
X_site_a = X_site_a.astype(np.float32)
X_site_b = X_site_b.astype(np.float32)
X_site_c = X_site_c.astype(np.float32)

y_site_a = y_site_a.astype(int)
y_site_b = y_site_b.astype(int)
y_site_c = y_site_c.astype(int)

In [18]:
print(f"Continuous feature shape: {X_site_a.shape}")
print("First few entries of each site have nan values due to feature engineering:\n", X_site_a[0])
print(X_site_a[ENTRIES_PER_DAY])

Continuous feature shape: (19345, 34)
First few entries of each site have nan values due to feature engineering:
 [ 0.57910997 -1.3638599  -0.5221737  -0.00325376 -0.00751908         nan
 -0.69390106  0.0624692  -0.54411376 -0.5452835          nan -0.64983785
 -0.84162885         nan -0.38954532 -0.25569385 -0.6574576  -0.75812143
 -1.602483    1.          0.          0.          0.          0.
  1.          0.          0.          0.          0.          0.
  0.          1.          0.5         0.8660254 ]
[ 3.1494236e-01 -1.3638599e+00 -5.2217370e-01 -3.2537556e-03
 -7.5190784e-03  9.0518305e-03 -3.7747535e-01  6.2469199e-02
 -5.4411376e-01 -5.4528350e-01  2.9363585e+00 -6.4983785e-01
 -8.4162885e-01  4.3428288e+00 -3.8954532e-01 -2.5569385e-01
 -6.5745759e-01 -7.5812143e-01 -1.6024830e+00  1.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  1.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00

In [19]:
np.unique(y_site_a)

array([0, 1, 2])

IMPORTANT: Remove entries, where features are incomplete (at start of dataset)

In [20]:
# Create mask to exclude first ENTRIES_PER_DAY of each site
mask_site_a = np.ones(len(X_site_a), dtype=bool)
mask_site_b = np.ones(len(X_site_b), dtype=bool)
mask_site_c = np.ones(len(X_site_c), dtype=bool)

# Site A: exclude indices 0 to ENTRIES_PER_DAY-1
mask_site_a[0:ENTRIES_PER_DAY] = False

# Site B: exclude indices (365*ENTRIES_PER_DAY) to (365*ENTRIES_PER_DAY + ENTRIES_PER_DAY-1)
mask_site_b[0:ENTRIES_PER_DAY] = False

# Site C: exclude indices (730*ENTRIES_PER_DAY) to (730*ENTRIES_PER_DAY + ENTRIES_PER_DAY-1)
mask_site_c[0:ENTRIES_PER_DAY] = False

# Apply mask to remove incomplete entries
X_site_a = X_site_a[mask_site_a]
X_site_b = X_site_b[mask_site_b]
X_site_c = X_site_c[mask_site_c]

y_site_a = y_site_a[mask_site_a]
y_site_b = y_site_b[mask_site_b]
y_site_c = y_site_c[mask_site_c]

### Data Splitting

In [21]:
# Calculate split indices (accounting for removed incomplete entries)
site_a_entries = (365 - 1) * ENTRIES_PER_DAY  # 364 days of site A
site_b_entries = (365 - 1) * ENTRIES_PER_DAY  # 364 days of site B

# Train on Site A and Site C, validate on Site B
X_train = np.vstack((X_site_a, X_site_c))
X_val = X_site_b
y_train = np.vstack((y_site_a, y_site_c))
y_val = y_site_b

In [22]:
print(len(X_train))
print(len(y_train))
print(len(X_val))
print(len(y_val))

38584
38584
19292
19292


## Prepare Sequences

In [23]:
config = {
    # Model hyperparameters
    'input_size': X_train.shape[1],
    'hidden_size': 64,          # TCN: number of channels per residual block
    'num_layers': 3,            # TCN: number of residual blocks (dilation levels)
    'kernel_size': 3,           # TCN: 1D conv kernel size
    'dropout': 0.3,
    'sequence_length': 12,

    # Training hyperparameters
    'learning_rate': 0.0001,
    'weight_decay': 1e-4,
    'batch_size': 32,
    'num_epochs': 50,
    'gradient_clip_val': 1.0,
    'warmup_epochs': 5,
    'optimizer': 'Adam',
    'loss_function': 'CrossEntropy',

    # Model architecture
    'model_type': 'TCN'         # 'LSTM' or 'TCN'
}


Create sequences to create a "sliding window" for the RNN architechture to predict the current hidden state based on the past values.

In [24]:
def create_sequences(X, y, seq_length=config['sequence_length']):
    sequences_X = []
    sequences_y = []
    
    for i in range(len(X) - seq_length):
        sequences_X.append(X[i:i+seq_length])
        sequences_y.append(y[i+seq_length])
    
    return np.array(sequences_X), np.array(sequences_y)

In [25]:
X_train_seq_a, y_train_seq_a = create_sequences(X_site_a, y_site_a)
X_train_seq_c, y_train_seq_c = create_sequences(X_site_c, y_site_c)
X_val_seq, y_val_seq = create_sequences(X_val, y_val)

X_train_seq = np.vstack((X_train_seq_a, X_train_seq_c))
y_train_seq = np.vstack((y_train_seq_a, y_train_seq_c))

In [26]:
print(f"Training sequences shape: {X_train_seq_a.shape}, Training targets shape: {y_train_seq_a.shape}")
print(f"Training sequences shape: {X_train_seq_c.shape}, Training targets shape: {y_train_seq_c.shape}")
print(f"Validation sequences shape: {X_val_seq.shape}, Validation targets shape: {y_val_seq.shape}")
print(f"Training sequences shape: {X_train_seq.shape}, Training targets shape: {y_train_seq.shape}")

Training sequences shape: (19280, 12, 34), Training targets shape: (19280, 1)
Training sequences shape: (19280, 12, 34), Training targets shape: (19280, 1)
Validation sequences shape: (19280, 12, 34), Validation targets shape: (19280, 1)
Training sequences shape: (38560, 12, 34), Training targets shape: (38560, 1)


In [27]:
np.unique(y_train_seq)

array([0, 1, 2])

Create DataLoader -> DataLoader's job: Efficiently load data in batches during training

In [28]:
train_dataset = FlextrackClassificationDataset(X_train_seq, y_train_seq)
val_dataset = FlextrackClassificationDataset(X_val_seq, y_val_seq)

batch_size = config['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

Create a simple RNN architecture.

Why not use nn.RNN: nn.RNN is just the recurrent layer - it's not a complete model. You need additional components to make predictions.

In [29]:
# --- Models ---------------------------------------------------------------
# This notebook originally used an LSTM. Below we add a Temporal Convolutional Network (TCN)
# classifier (causal, dilated 1D convs with residual connections).
#
# Expected input shape for BOTH models: (batch, seq_len, num_features)

class Chomp1d(nn.Module):
    """Remove extra right-padding so output length matches input length."""
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        # x: (batch, channels, length)
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation  # causal padding on the left via Conv1d padding + chomp

        self.conv1 = nn.utils.weight_norm(
            nn.Conv1d(in_channels, out_channels, kernel_size,
                      padding=padding, dilation=dilation)
        )
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.utils.weight_norm(
            nn.Conv1d(out_channels, out_channels, kernel_size,
                      padding=padding, dilation=dilation)
        )
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)

        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.relu1, self.drop1,
            self.conv2, self.chomp2, self.relu2, self.drop2
        )

        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.final_relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.final_relu(out + res)


class TCNEncoder(nn.Module):
    """Stack of TemporalBlocks with exponentially increasing dilation."""
    def __init__(self, input_channels, channels, kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        in_ch = input_channels
        for i, out_ch in enumerate(channels):
            dilation = 2 ** i
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size, dilation, dropout))
            in_ch = out_ch
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # x: (batch, channels, length)
        return self.network(x)


class TCNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, kernel_size=3, num_classes=3, dropout=0.2):
        super().__init__()
        # Use a constant number of channels per block (hidden_size) across num_layers blocks.
        channels = [hidden_size] * num_layers
        self.tcn = TCNEncoder(input_channels=input_size, channels=channels, kernel_size=kernel_size, dropout=dropout)

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        # x: (batch, seq_len, input_size)  -> Conv1d expects (batch, channels, length)
        x = x.transpose(1, 2)  # (batch, input_size, seq_len)
        y = self.tcn(x)        # (batch, hidden_size, seq_len)
        last = y[:, :, -1]     # last time step representation (causal)
        return self.head(last)


Define cuda as the device to make the training possible to the GPU.

In [30]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Sweep

In [31]:
sweep_config = {
    "method": "bayes",
    "metric": {
        "name": "val/f1",
        "goal": "maximize"
    },

    "parameters": {
        # ---------------------------------------------------
        # Tunable hyperparameters
        # ---------------------------------------------------
        "learning_rate": {
            "min": 1e-5,
            "max": 5e-3
        },
        # For TCN, hidden_size = number of channels per residual block
        "hidden_size": {
            "values": [32, 64, 96, 128, 192]
        },
        # For TCN, num_layers = number of residual blocks (dilation levels)
        "num_layers": {
            "values": [2, 3, 4, 5]
        },
        "kernel_size": {
            "values": [2, 3, 5, 7]
        },
        "dropout": {
            "min": 0.0,
            "max": 0.5
        },
        "weight_decay": {
            "values": [0.0, 1e-5, 1e-4, 1e-3]
        },

        # ---------------------------------------------------
        # Fixed hyperparameters (kept constant from your config)
        # ---------------------------------------------------
        "batch_size": {
            "value": 32
        },
        "input_size": {
            "value": X_train.shape[1]
        },
        "sequence_length": {
            "value": 12
        },
        "num_epochs": {
            "value": 50
        },
        "gradient_clip_val": {
            "value": 1.0
        },
        "warmup_epochs": {
            "value": 5
        },
        "optimizer": {
            "value": "Adam"
        },
        "loss_function": {
            "value": "CrossEntropy"
        },
        "num_classes": {
            "value": 3
        },
        "model_type": {
            "value": "TCN"  # set to "LSTM" to sweep the original model
        }
    }
}
sweep_id = wandb.sweep(sweep_config, project="AICOMP_Flextrack")


Create sweep with ID: kizwty9z
Sweep URL: https://wandb.ai/fabian-dubach-hochschule-luzern/AICOMP_Flextrack/sweeps/kizwty9z


# Training

In [32]:
# Warmup scheduler function
def get_lr_with_warmup(epoch, base_lr, warmup_epochs):
    """
    Calculate learning rate with linear warmup
    
    Args:
        epoch: Current epoch (0-indexed)
        base_lr: Target learning rate after warmup
        warmup_epochs: Number of epochs for warmup
    
    Returns:
        Current learning rate
    """
    if warmup_epochs == 0 or epoch >= warmup_epochs:
        return base_lr
    else:
        # Linear warmup from 0 to base_lr
        return base_lr * (epoch + 1) / warmup_epochs

In [33]:
def train():
    wandb.init(
        project="AICOMP_Flextrack",
        entity="fabian-dubach-hochschule-luzern"
    )

    config = wandb.config

    # Auto-name the run based on sweep parameters
    wandb.run.name = f"{config.model_type}-classification-hs_{config.hidden_size}-nl_{config.num_layers}-ks_{getattr(config, 'kernel_size', 'na')}-lr_{config.learning_rate:.0e}"

    print("WandB initialized successfully!")

    # Choose model based on config.model_type
    if config.model_type.upper() == 'TCN':
        model = TCNClassifier(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            num_layers=config.num_layers,
            kernel_size=getattr(config, 'kernel_size', 3),
            dropout=config.dropout,
        ).to(device)
    else:
        raise ValueError('Dont use other models than TCN in this notebook!')
    print(f"Model architecture:\n{model}")

    # Compute class weights from training set
    counts = np.bincount(y_train_seq.flatten())
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.mean()
    weights = np.clip(weights, 0.1, 10.0)
    class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')

    print("Starting training...")
    print(f"Warmup enabled: {config.warmup_epochs} epochs")

    for epoch in range(config.num_epochs):

        # Learning rate warmup
        current_lr = get_lr_with_warmup(epoch, config.learning_rate, config.warmup_epochs)
        for param_group in optimizer.param_groups:
            param_group["lr"] = current_lr

        # Training
        model.train()
        train_loss = 0
        train_preds = []
        train_targets = []

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            clip_grad_norm_(model.parameters(), config.gradient_clip_val)
            optimizer.step()

            train_loss += loss.item()
            train_preds.append(torch.argmax(logits, dim=1).cpu().numpy())
            train_targets.append(y_batch.cpu().numpy())

        train_loss /= len(train_loader)
        train_preds = np.concatenate(train_preds)
        train_targets = np.concatenate(train_targets)
        train_gmean = geometric_mean_score(train_targets, train_preds)
        train_f1 = f1_score(train_targets, train_preds, average="macro")

        # Validation
        model.eval()
        val_loss = 0
        val_preds = []
        val_targets = []

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_loss += loss.item()

                val_preds.append(torch.argmax(logits, dim=1).cpu().numpy())
                val_targets.append(y_batch.cpu().numpy())

        val_loss /= len(val_loader)
        val_preds = np.concatenate(val_preds)
        val_targets = np.concatenate(val_targets)
        val_gmean = geometric_mean_score(val_targets, val_preds)
        val_f1 = f1_score(val_targets, val_preds, average="macro")

        # Log to W&B
        wandb.log({
            "epoch": epoch,
            "lr": current_lr,
            "train/loss": train_loss,
            "val/loss": val_loss,
            "train/geometric_mean": train_gmean,
            "train/f1": train_f1,
            "val/geometric_mean": val_gmean,
            "val/f1": val_f1
        })

        # Print progress occasionally
        if (epoch + 1) % 10 == 0:
            print(f"\nEpoch [{epoch+1}/{config.num_epochs}]")
            print(f"LR: {current_lr:.6f}")
            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            print(f"Train - GMean: {train_gmean:.4f} | F1: {train_f1:.4f}")
            print(f"Val   - GMean: {val_gmean:.4f} | F1: {val_f1:.4f}")

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pt")

    wandb.finish()
    print("\nTraining completed!")

In [34]:
wandb.agent(sweep_id, function=train, count=20)

wandb: Agent Starting Run: c8gta6cf with config:
wandb: 	batch_size: 32
wandb: 	dropout: 0.2249943467321111
wandb: 	gradient_clip_val: 1
wandb: 	hidden_size: 192
wandb: 	input_size: 34
wandb: 	kernel_size: 5
wandb: 	learning_rate: 0.004688859254245542
wandb: 	loss_function: CrossEntropy
wandb: 	model_type: TCN
wandb: 	num_classes: 3
wandb: 	num_epochs: 50
wandb: 	num_layers: 4
wandb: 	optimizer: Adam
wandb: 	sequence_length: 12
wandb: 	warmup_epochs: 5
wandb: 	weight_decay: 1e-05


WandB initialized successfully!


c:\Users\timon\.pyenv-win-venv\envs\aicomp\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Model architecture:
TCNClassifier(
  (tcn): TCNEncoder(
    (network): Sequential(
      (0): TemporalBlock(
        (conv1): Conv1d(34, 192, kernel_size=(5,), stride=(1,), padding=(4,))
        (chomp1): Chomp1d()
        (relu1): ReLU()
        (drop1): Dropout(p=0.2249943467321111, inplace=False)
        (conv2): Conv1d(192, 192, kernel_size=(5,), stride=(1,), padding=(4,))
        (chomp2): Chomp1d()
        (relu2): ReLU()
        (drop2): Dropout(p=0.2249943467321111, inplace=False)
        (net): Sequential(
          (0): Conv1d(34, 192, kernel_size=(5,), stride=(1,), padding=(4,))
          (1): Chomp1d()
          (2): ReLU()
          (3): Dropout(p=0.2249943467321111, inplace=False)
          (4): Conv1d(192, 192, kernel_size=(5,), stride=(1,), padding=(4,))
          (5): Chomp1d()
          (6): ReLU()
          (7): Dropout(p=0.2249943467321111, inplace=False)
        )
        (downsample): Conv1d(34, 192, kernel_size=(1,), stride=(1,))
        (final_relu): ReLU()
    

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
lr,▁▃▄▆████████████████████████████████████
train/f1,▂▁▁▂▄█▃▄▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/geometric_mean,▃▁▁▁▆█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,▇▂▁▆▃▆█▇█▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
val/f1,▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/geometric_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,▃▆▆▃▄█▁▁▂▁▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
epoch,49
lr,0.00469
train/f1,0.32446


...

WandB initialized successfully!


c:\Users\timon\.pyenv-win-venv\envs\aicomp\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Model architecture:
TCNClassifier(
  (tcn): TCNEncoder(
    (network): Sequential(
      (0): TemporalBlock(
        (conv1): Conv1d(34, 128, kernel_size=(7,), stride=(1,), padding=(6,))
        (chomp1): Chomp1d()
        (relu1): ReLU()
        (drop1): Dropout(p=0.21147992842625035, inplace=False)
        (conv2): Conv1d(128, 128, kernel_size=(7,), stride=(1,), padding=(6,))
        (chomp2): Chomp1d()
        (relu2): ReLU()
        (drop2): Dropout(p=0.21147992842625035, inplace=False)
        (net): Sequential(
          (0): Conv1d(34, 128, kernel_size=(7,), stride=(1,), padding=(6,))
          (1): Chomp1d()
          (2): ReLU()
          (3): Dropout(p=0.21147992842625035, inplace=False)
          (4): Conv1d(128, 128, kernel_size=(7,), stride=(1,), padding=(6,))
          (5): Chomp1d()
          (6): ReLU()
          (7): Dropout(p=0.21147992842625035, inplace=False)
        )
        (downsample): Conv1d(34, 128, kernel_size=(1,), stride=(1,))
        (final_relu): ReLU()


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
lr,▁▃▄▆████████████████████████████████████
train/f1,▁▁▁▁▁▁▁▁▂▂▃▃▄▄▄▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████
train/geometric_mean,▁▁▁▁▁▁▁▁▁▂▃▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇██████
train/loss,█▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val/f1,▁▁▁▁▁▁▂▃▃▄▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇████▇█▇▇███
val/geometric_mean,▁▁▁▁▁▁▁▁▁▃▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██▇█████▇▇███
val/loss,▂▂▂▂▂▂▂▁▁▁▁▁▁▂▁▁▂▂▂▃▄▄▄▅▄▅▅▅█▇▅▅▇▇▆█▇▄▄▆
epoch,49
lr,0.00021
train/f1,0.86065



Training completed!
